# Error Analysis

Systematic error analysis across all models (decoders: zero-shot & chain-of-thought; encoders: fine-tuned) on two datasets (**CSC** and **MultiPICo**) for two input types: `last_u1_message` and `conversation`.

Analysis covers:
1. **Data loading** — collect all predictions into a unified DataFrame
2. **All-models-wrong** — examples where every model fails (both input types)
3. **Input-type disagreement** — examples where models fail on `last_u1_message` but not `conversation` (and vice-versa)
4. **Error rate by label** — bias towards false positives / false negatives
5. **Model-specific error patterns** — which model makes unique errors
6. **Disagreement matrix** — pairwise model disagreement
7. **Qualitative inspection** — display dialogue text for hard cases


In [1]:
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations

BASE = Path(r"c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\output")

INPUT_TYPES = ["last_u1_message", "conversation"]
DATASETS   = ["CSC", "MultiPICo"]

# ── DECODER files ─────────────────────────────────────────────────────────────
DECODERS = {
    "Gemma-2-9b":    BASE / "decoders" / "google_gemma-2-9b-it",
    "Ministral-8B":  BASE / "decoders" / "mistralai_Ministral-8B-Instruct-2410",
    "Qwen-2.5-7B":   BASE / "decoders" / "Qwen_Qwen2.5-7B-Instruct",
}
DECODER_PROMPTS = ["zero_shot", "chain_of_thought"]

# ── ENCODER files ─────────────────────────────────────────────────────────────
ENCODERS = {
    "XLM-RoBERTa-large": BASE / "encoders" / "FacebookAI_xlm-roberta-large",
    "EuroBERT-210m":     BASE / "encoders" / "EuroBERT_EuroBERT-210m",
}

# ──────────────────────────────────────────────────────────────────────────────
records = []

for model_name, folder in DECODERS.items():
    for prompt in DECODER_PROMPTS:
        for dataset in DATASETS:
            for inp in INPUT_TYPES:
                # find matching file
                candidates = list(folder.glob(f"{prompt}_*_{dataset}_{inp}.csv"))
                if not candidates:
                    continue
                df = pd.read_csv(candidates[0], usecols=["unique_id", "model_answer", "label"])
                df = df[df["model_answer"].notna()]
                df["model"]      = f"{model_name}_{prompt}"
                df["dataset"]    = dataset
                df["input_type"] = inp
                records.append(df)

for model_name, enc_base in ENCODERS.items():
    for dataset in DATASETS:
        for inp in INPUT_TYPES:
            pred_dir = enc_base / dataset / inp / "predictions"
            candidates = list(pred_dir.glob("*.csv"))
            if not candidates:
                continue
            df = pd.read_csv(candidates[0], usecols=["unique_id", "model_answer", "label"])
            df = df[df["model_answer"].notna()]
            df["model"]      = f"{model_name}_fine_tuning"
            df["dataset"]    = dataset
            df["input_type"] = inp
            records.append(df)

all_preds = pd.concat(records, ignore_index=True)
all_preds["model_answer"] = pd.to_numeric(all_preds["model_answer"], errors="coerce")
all_preds["label"]        = pd.to_numeric(all_preds["label"],        errors="coerce")
all_preds = all_preds.dropna(subset=["model_answer", "label"])
all_preds["model_answer"] = all_preds["model_answer"].astype(int)
all_preds["label"]        = all_preds["label"].astype(int)
all_preds["correct"]      = (all_preds["model_answer"] == all_preds["label"]).astype(int)
all_preds["error_type"]   = all_preds.apply(
    lambda r: "FP" if r["model_answer"] == 1 and r["label"] == 0
         else ("FN" if r["model_answer"] == 0 and r["label"] == 1 else "OK"),
    axis=1
)

models = all_preds["model"].unique().tolist()
print(f"Models loaded: {models}")
print(f"Total prediction rows: {len(all_preds)}")
all_preds.head(3)


Models loaded: ['Gemma-2-9b_zero_shot', 'Gemma-2-9b_chain_of_thought', 'Ministral-8B_zero_shot', 'Ministral-8B_chain_of_thought', 'Qwen-2.5-7B_zero_shot', 'Qwen-2.5-7B_chain_of_thought', 'XLM-RoBERTa-large_fine_tuning', 'EuroBERT-210m_fine_tuning']
Total prediction rows: 10288


,unique_id,label,model_answer,model,dataset,input_type,correct,error_type
0,109_1043,0,1,Gemma-2-9b_zero_shot,CSC,last_u1_message,0,FP
1,116_1017,0,1,Gemma-2-9b_zero_shot,CSC,last_u1_message,0,FP
2,126_1030,0,0,Gemma-2-9b_zero_shot,CSC,last_u1_message,1,OK


## 1. Overall accuracy by model, dataset, input type

In [2]:
acc_table = (
    all_preds.groupby(["dataset", "input_type", "model"])["correct"]
    .mean()
    .round(3)
    .reset_index()
    .rename(columns={"correct": "accuracy"})
    .pivot_table(index=["dataset", "model"], columns="input_type", values="accuracy")
    .reset_index()
)
print(acc_table.to_string(index=False))


  dataset                         model  conversation  last_u1_message
      CSC     EuroBERT-210m_fine_tuning         0.695            0.406
      CSC   Gemma-2-9b_chain_of_thought         0.397            0.388
      CSC          Gemma-2-9b_zero_shot         0.456            0.424
      CSC Ministral-8B_chain_of_thought         0.415            0.447
      CSC        Ministral-8B_zero_shot         0.484            0.440
      CSC  Qwen-2.5-7B_chain_of_thought         0.443            0.562
      CSC         Qwen-2.5-7B_zero_shot         0.500            0.511
      CSC XLM-RoBERTa-large_fine_tuning         0.709            0.328
MultiPICo     EuroBERT-210m_fine_tuning         0.652            0.594
MultiPICo   Gemma-2-9b_chain_of_thought         0.333            0.382
MultiPICo          Gemma-2-9b_zero_shot         0.454            0.406
MultiPICo Ministral-8B_chain_of_thought         0.401            0.406
MultiPICo        Ministral-8B_zero_shot         0.444            0.440
MultiP

## 2. Error-type breakdown (FP / FN) per model × dataset × input_type

In [3]:
error_breakdown = (
    all_preds[all_preds["error_type"] != "OK"]
    .groupby(["dataset", "input_type", "model", "error_type"])
    .size()
    .reset_index(name="count")
)
pivot_err = error_breakdown.pivot_table(
    index=["dataset", "input_type", "model"],
    columns="error_type",
    values="count",
    fill_value=0
).reset_index()
print(pivot_err.to_string(index=False))


  dataset      input_type                         model   FN    FP
      CSC    conversation     EuroBERT-210m_fine_tuning 43.0  90.0
      CSC    conversation   Gemma-2-9b_chain_of_thought 11.0 252.0
      CSC    conversation          Gemma-2-9b_zero_shot 16.0 221.0
      CSC    conversation Ministral-8B_chain_of_thought 18.0 237.0
      CSC    conversation        Ministral-8B_zero_shot 28.0 197.0
      CSC    conversation  Qwen-2.5-7B_chain_of_thought 26.0 217.0
      CSC    conversation         Qwen-2.5-7B_zero_shot 33.0 185.0
      CSC    conversation XLM-RoBERTa-large_fine_tuning 43.0  84.0
      CSC last_u1_message     EuroBERT-210m_fine_tuning 12.0 247.0
      CSC last_u1_message   Gemma-2-9b_chain_of_thought  8.0 259.0
      CSC last_u1_message          Gemma-2-9b_zero_shot 24.0 227.0
      CSC last_u1_message Ministral-8B_chain_of_thought 36.0 205.0
      CSC last_u1_message        Ministral-8B_zero_shot 40.0 204.0
      CSC last_u1_message  Qwen-2.5-7B_chain_of_thought 63.0 1

## 3. Cases where ALL models are wrong

For each (dataset, input_type) pair, find examples where every model predicts incorrectly.

In [4]:
all_wrong_by_setting = {}

for dataset in DATASETS:
    for inp in INPUT_TYPES:
        subset = all_preds[(all_preds["dataset"] == dataset) & (all_preds["input_type"] == inp)]
        n_models = subset["model"].nunique()
        # count how many models got each example wrong
        wrong_counts = (
            subset[subset["correct"] == 0]
            .groupby("unique_id")["model"]
            .nunique()
            .reset_index(name="n_models_wrong")
        )
        # keep label and total-model count
        label_map = subset[["unique_id", "label"]].drop_duplicates()
        wrong_counts = wrong_counts.merge(label_map, on="unique_id")
        all_wrong = wrong_counts[wrong_counts["n_models_wrong"] == n_models].copy()
        all_wrong["dataset"]    = dataset
        all_wrong["input_type"] = inp
        all_wrong["n_total_models"] = n_models
        key = f"{dataset}_{inp}"
        all_wrong_by_setting[key] = all_wrong
        print(f"[{key}]  total models={n_models}  examples where ALL wrong: {len(all_wrong)}")
        print(f"  label=0 (non-irony): {(all_wrong['label']==0).sum()}   label=1 (irony): {(all_wrong['label']==1).sum()}")


[CSC_last_u1_message]  total models=8  examples where ALL wrong: 67
  label=0 (non-irony): 67   label=1 (irony): 0
[CSC_conversation]  total models=8  examples where ALL wrong: 24
  label=0 (non-irony): 24   label=1 (irony): 0
[MultiPICo_last_u1_message]  total models=8  examples where ALL wrong: 20
  label=0 (non-irony): 20   label=1 (irony): 0
[MultiPICo_conversation]  total models=8  examples where ALL wrong: 18
  label=0 (non-irony): 18   label=1 (irony): 0


## 4. Input-type disagreement: wrong on `last_u1_message` but correct on `conversation` (and vice-versa)

For each model separately, find examples where the model is wrong in one input setting but right in the other.
Then aggregate to find *consistent* patterns across models.

In [5]:
# pivot so each row = (model, dataset, unique_id) with correctness per input type
pivot_inp = all_preds.pivot_table(
    index=["model", "dataset", "unique_id", "label"],
    columns="input_type",
    values="correct"
).reset_index()

# keep only rows where both input types are available for that model
pivot_inp = pivot_inp.dropna(subset=INPUT_TYPES)
pivot_inp[INPUT_TYPES] = pivot_inp[INPUT_TYPES].astype(int)

# wrong on last_u1 but right on conversation → context HELPS
pivot_inp["lu1_wrong_conv_ok"]  = ((pivot_inp["last_u1_message"] == 0) & (pivot_inp["conversation"] == 1)).astype(int)
# right on last_u1 but wrong on conversation → context HURTS
pivot_inp["lu1_ok_conv_wrong"]  = ((pivot_inp["last_u1_message"] == 1) & (pivot_inp["conversation"] == 0)).astype(int)
# wrong on BOTH
pivot_inp["both_wrong"]         = ((pivot_inp["last_u1_message"] == 0) & (pivot_inp["conversation"] == 0)).astype(int)

for dataset in DATASETS:
    sub = pivot_inp[pivot_inp["dataset"] == dataset]
    print(f"\n=== {dataset} ===")
    summary = sub.groupby("model")[["lu1_wrong_conv_ok", "lu1_ok_conv_wrong", "both_wrong"]].sum()
    print(summary.to_string())



=== CSC ===
input_type                     lu1_wrong_conv_ok  lu1_ok_conv_wrong  both_wrong
model                                                                          
EuroBERT-210m_fine_tuning                    162                 36          97
Gemma-2-9b_chain_of_thought                   41                 37         226
Gemma-2-9b_zero_shot                          65                 51         186
Ministral-8B_chain_of_thought                 63                 77         178
Ministral-8B_zero_shot                        98                 79         146
Qwen-2.5-7B_chain_of_thought                  71                123         120
Qwen-2.5-7B_zero_shot                         62                 67         151
XLM-RoBERTa-large_fine_tuning                209                 43          84

=== MultiPICo ===
input_type                     lu1_wrong_conv_ok  lu1_ok_conv_wrong  both_wrong
model                                                                          
EuroBERT

### 4b. Aggregate across models: how many models show each disagreement pattern?

Find examples where the *majority* of models (or all models) show `lu1_wrong_conv_ok` vs `lu1_ok_conv_wrong`.

In [6]:
agg_disagree = (
    pivot_inp.groupby(["dataset", "unique_id", "label"])[
        ["lu1_wrong_conv_ok", "lu1_ok_conv_wrong", "both_wrong"]
    ]
    .sum()
    .reset_index()
)

n_total_models = pivot_inp["model"].nunique()

# "context helps": majority of models are wrong on last_u1 but right on conversation
context_helps = agg_disagree[agg_disagree["lu1_wrong_conv_ok"] > n_total_models / 2].copy()
# "context hurts": majority of models are right on last_u1 but wrong on conversation
context_hurts = agg_disagree[agg_disagree["lu1_ok_conv_wrong"] > n_total_models / 2].copy()

for dataset in DATASETS:
    ch = context_helps[context_helps["dataset"] == dataset]
    cu = context_hurts[context_hurts["dataset"] == dataset]
    print(f"\n=== {dataset} (total models={n_total_models}) ===")
    print(f"  'Context helps' (majority wrong on last_u1 but right on conv): {len(ch)} examples")
    print(f"    label=0: {(ch['label']==0).sum()}  label=1: {(ch['label']==1).sum()}")
    print(f"  'Context hurts' (majority right on last_u1 but wrong on conv): {len(cu)} examples")
    print(f"    label=0: {(cu['label']==0).sum()}  label=1: {(cu['label']==1).sum()}")



=== CSC (total models=8) ===
  'Context helps' (majority wrong on last_u1 but right on conv): 13 examples
    label=0: 8  label=1: 5
  'Context hurts' (majority right on last_u1 but wrong on conv): 9 examples
    label=0: 8  label=1: 1

=== MultiPICo (total models=8) ===
  'Context helps' (majority wrong on last_u1 but right on conv): 1 examples
    label=0: 1  label=1: 0
  'Context hurts' (majority right on last_u1 but wrong on conv): 4 examples
    label=0: 2  label=1: 2


## 5. Model-unique errors (errors made by only one model)

Which model makes errors that no other model makes? These are model-specific weaknesses.

In [7]:
for dataset in DATASETS:
    for inp in INPUT_TYPES:
        sub = all_preds[(all_preds["dataset"] == dataset) & (all_preds["input_type"] == inp)]
        # per example: count how many models got it wrong
        wrong_counts = (
            sub[sub["correct"] == 0]
            .groupby("unique_id")["model"]
            .nunique()
            .reset_index(name="n_wrong_models")
        )
        # unique errors = only 1 model wrong
        unique_errors = wrong_counts[wrong_counts["n_wrong_models"] == 1]["unique_id"].tolist()
        # which model is responsible?
        resp = (
            sub[(sub["correct"] == 0) & (sub["unique_id"].isin(unique_errors))]
            .groupby("model")["unique_id"]
            .count()
            .sort_values(ascending=False)
        )
        print(f"\n[{dataset} | {inp}]  unique errors per model:")
        print(resp.to_string())



[CSC | last_u1_message]  unique errors per model:
model
Ministral-8B_zero_shot           13
Qwen-2.5-7B_chain_of_thought     12
XLM-RoBERTa-large_fine_tuning     5
Qwen-2.5-7B_zero_shot             5
EuroBERT-210m_fine_tuning         4
Ministral-8B_chain_of_thought     2
Gemma-2-9b_zero_shot              1
Gemma-2-9b_chain_of_thought       1

[CSC | conversation]  unique errors per model:
model
EuroBERT-210m_fine_tuning        7
Gemma-2-9b_chain_of_thought      7
XLM-RoBERTa-large_fine_tuning    7
Ministral-8B_zero_shot           7
Ministral-8B_chain_of_thought    5
Qwen-2.5-7B_chain_of_thought     4
Qwen-2.5-7B_zero_shot            4
Gemma-2-9b_zero_shot             1

[MultiPICo | last_u1_message]  unique errors per model:
model
EuroBERT-210m_fine_tuning        13
Qwen-2.5-7B_zero_shot             4
Qwen-2.5-7B_chain_of_thought      4
XLM-RoBERTa-large_fine_tuning     4
Ministral-8B_chain_of_thought     2
Gemma-2-9b_zero_shot              1
Ministral-8B_zero_shot            1

[Mult

## 6. Pairwise model disagreement matrix

For each pair of models (same dataset + input_type), what fraction of examples do they predict differently?

In [8]:
for dataset in DATASETS:
    for inp in INPUT_TYPES:
        sub = all_preds[(all_preds["dataset"] == dataset) & (all_preds["input_type"] == inp)]
        wide = sub.pivot_table(index="unique_id", columns="model", values="model_answer")
        wide = wide.dropna()
        model_cols = wide.columns.tolist()
        rows = []
        for m1, m2 in combinations(model_cols, 2):
            disagree_rate = (wide[m1] != wide[m2]).mean()
            rows.append({"model_A": m1, "model_B": m2, "disagree_rate": round(disagree_rate, 3)})
        df_dis = pd.DataFrame(rows).sort_values("disagree_rate", ascending=False)
        print(f"\n=== {dataset} | {inp} ===")
        print(df_dis.to_string(index=False))



=== CSC | last_u1_message ===
                      model_A                       model_B  disagree_rate
 Qwen-2.5-7B_chain_of_thought XLM-RoBERTa-large_fine_tuning          0.523
    EuroBERT-210m_fine_tuning  Qwen-2.5-7B_chain_of_thought          0.495
       Ministral-8B_zero_shot  Qwen-2.5-7B_chain_of_thought          0.452
  Gemma-2-9b_chain_of_thought  Qwen-2.5-7B_chain_of_thought          0.450
         Gemma-2-9b_zero_shot  Qwen-2.5-7B_chain_of_thought          0.390
Ministral-8B_chain_of_thought  Qwen-2.5-7B_chain_of_thought          0.385
        Qwen-2.5-7B_zero_shot XLM-RoBERTa-large_fine_tuning          0.381
    EuroBERT-210m_fine_tuning         Qwen-2.5-7B_zero_shot          0.376
       Ministral-8B_zero_shot         Qwen-2.5-7B_zero_shot          0.356
Ministral-8B_chain_of_thought        Ministral-8B_zero_shot          0.323
    EuroBERT-210m_fine_tuning        Ministral-8B_zero_shot          0.323
  Gemma-2-9b_chain_of_thought         Qwen-2.5-7B_zero_shot          

## 7. Qualitative inspection — load original dialogues and display hard cases

Load the original unified corpus to attach the actual dialogue text to the hard examples.

In [9]:
import json

CORPUS_ROOT = Path(r"c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\datasets")

def load_json_dataset(path):
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    entries = data["entries"] if "entries" in data else data
    rows = []
    for e in entries:
        row = {
            "unique_id": e.get("unique_id"),
            "label":     e.get("label"),
            "post":      e.get("post"),
            "reply":     e.get("reply"),
            "context":   e.get("context"),
            "dialogue":  str(e.get("dialogue", "")),
            "source":    e.get("source"),
        }
        rows.append(row)
    return pd.DataFrame(rows)

csc_df       = load_json_dataset(CORPUS_ROOT / "CSC_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_final.json")
multipico_df = load_json_dataset(CORPUS_ROOT / "MultiPICo_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_final.json")

corpus = {"CSC": csc_df, "MultiPICo": multipico_df}
print("CSC columns:",       csc_df.columns.tolist(),       "  rows:", len(csc_df))
print("MultiPICo columns:", multipico_df.columns.tolist(), "  rows:", len(multipico_df))


CSC columns: ['unique_id', 'label', 'post', 'reply', 'context', 'dialogue', 'source']   rows: 2905
MultiPICo columns: ['unique_id', 'label', 'post', 'reply', 'context', 'dialogue', 'source']   rows: 1377


In [10]:
def show_examples(uid_list, dataset_name, n=5, title=""):
    """Print dialogue text for a list of unique_ids."""
    df_src = corpus[dataset_name]
    # find text columns
    text_cols = [c for c in df_src.columns if c.lower() in ("post", "reply", "dialogue", "context", "text")]
    sample = uid_list[:n]
    print(f"\n{'='*70}")
    print(f"  {title}  [{dataset_name}]  showing {len(sample)}/{len(uid_list)}")
    print(f"{'='*70}")
    for uid in sample:
        row = df_src[df_src["unique_id"] == uid]
        if row.empty:
            print(f"\n  [{uid}]  (not found in corpus)")
            continue
        row = row.iloc[0]
        print(f"\n  ID: {uid}   label={row.get('label', '?')}")
        for c in text_cols:
            val = row.get(c, "")
            if pd.notna(val) and str(val).strip():
                print(f"  {c}: {str(val)[:300]}")
        print()


### 7a. All-models-wrong examples (conversation input)

In [11]:
for dataset in DATASETS:
    key = f"{dataset}_conversation"
    if key not in all_wrong_by_setting:
        continue
    uids = all_wrong_by_setting[key]["unique_id"].tolist()
    show_examples(uids, dataset, n=5, title=f"ALL MODELS WRONG | conversation")



  ALL MODELS WRONG | conversation  [CSC]  showing 5/24

  ID: 101_1055   label=None
  reply: Steve you need to do what you said you will do, sports is always going to be here, no get up and workout.
  context: Steve's new year's resolution was to learn French and go to the gym every day. That was three weeks ago.  At the moment, instead of that, he is watching sports and eating popcorn on the couch.
  dialogue: {'turns': [{'u1': "Hey buddy, I'm really struggling to get into this French thing. I've been watching some YouTube videos but I just can't seem to pick it up.", 'u2': "Come on, you said you were going to learn French. It's not that hard. Just put in the time and you'll get it."}, {'u1': "But the gym


  ID: 108_1048   label=None
  reply: get off the wheel, i am taking over
  context: Steve and you are driving in town. Steve is the driver. You notice that he keeps making abrupt stops for no reason. Steve has never been an excellent driver but he hasn't been this bad, either.
  d

### 7b. "Context helps" examples — models wrong on last_u1 but right with full conversation

In [12]:
for dataset in DATASETS:
    uids = context_helps[context_helps["dataset"] == dataset]["unique_id"].tolist()
    show_examples(uids, dataset, n=5, title="CONTEXT HELPS (lu1 wrong → conv right)")



  CONTEXT HELPS (lu1 wrong → conv right)  [CSC]  showing 5/13

  ID: 104_1023   label=None
  reply: Who are you expecting to meet at the wedding,  A girl ?  ... l really wish we were not going to this wedding
  context: You and Steve have a mutual friend. She is getting married in two weeks. You are both invited to the wedding. You hate weddings but Steve seems to be pretty excited about potentially meeting somebody there. He is already picking out his suit and making a fuss.
  dialogue: {'turns': [{'u1': 'Have you thought about the wedding invitation we got? I was thinking it would be a great opportunity for me to meet some new people.', 'u2': "Yeah right, who are you trying to impress with that suit you're picking out?"}, {'u1': 'A girl, maybe. It would be awesome if I could meet 


  ID: 121_1007   label=None
  reply: The puppy is great lots of fun
  context: You were watching Steve's new puppy for a week while he was traveling in Italy. The puppy made a total mess out of your apar

### 7c. "Context hurts" examples — models right on last_u1 but wrong with full conversation

In [13]:
for dataset in DATASETS:
    uids = context_hurts[context_hurts["dataset"] == dataset]["unique_id"].tolist()
    show_examples(uids, dataset, n=5, title="CONTEXT HURTS (lu1 right → conv wrong)")



  CONTEXT HURTS (lu1 right → conv wrong)  [CSC]  showing 5/9

  ID: 112_1014   label=None
  reply: Why did you agree to it then?
  context: Steve has to play soccer for a fundraising event he is a part of. He has always been bad at sports. He is complaining about having to play soccer soon.
  dialogue: {'turns': [{'u1': "I don't know, I guess I just really wanted to help out. The school's been good to me, so I figured it would be a small price to pay.", 'u2': "Dude, you know you're going to embarrass me in front of all the teachers and the PTA, don't you?"}, {'u1': "Haha, I guess so. But it's not 


  ID: 227_2098   label=None
  reply: Sure buddy, but are you going to pony up the dough?  Pretty sure I'm still waiting for you to pay me back from lunch the other day.  Tell you what, how about I see how much it's going to cost on the app, you sling me that much, and then I'll complete the booking.  Sound fair?
  context: Steve wants to take a cab to go home. He asks you to call a taxi fo

## 8. Export hard-cases to CSV for manual inspection in the paper

In [14]:
OUT_DIR = Path(r"c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output")
OUT_DIR.mkdir(exist_ok=True)

def enrich_and_save(uid_df, dataset_name, filename):
    """Merge with corpus text columns and save."""
    df_src = corpus[dataset_name]
    text_cols = [c for c in df_src.columns if c.lower() in ("post", "reply", "dialogue", "context", "label", "unique_id")]
    merged = uid_df.merge(df_src[text_cols].drop_duplicates("unique_id"), on="unique_id", how="left")
    path = OUT_DIR / filename
    merged.to_csv(path, index=False)
    print(f"Saved {len(merged)} rows → {path}")
    return merged

for dataset in DATASETS:
    # all-models-wrong per input type
    for inp in INPUT_TYPES:
        key = f"{dataset}_{inp}"
        if key in all_wrong_by_setting and not all_wrong_by_setting[key].empty:
            enrich_and_save(
                all_wrong_by_setting[key][["unique_id", "label", "n_models_wrong"]],
                dataset,
                f"all_wrong_{dataset}_{inp}.csv"
            )

# context helps / hurts
for dataset in DATASETS:
    ch = context_helps[context_helps["dataset"] == dataset][["unique_id", "label", "lu1_wrong_conv_ok"]]
    if not ch.empty:
        enrich_and_save(ch, dataset, f"context_helps_{dataset}.csv")

    cu = context_hurts[context_hurts["dataset"] == dataset][["unique_id", "label", "lu1_ok_conv_wrong"]]
    if not cu.empty:
        enrich_and_save(cu, dataset, f"context_hurts_{dataset}.csv")

print("\nAll exports done.")


Saved 67 rows → c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\all_wrong_CSC_last_u1_message.csv
Saved 24 rows → c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\all_wrong_CSC_conversation.csv
Saved 20 rows → c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\all_wrong_MultiPICo_last_u1_message.csv
Saved 18 rows → c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\all_wrong_MultiPICo_conversation.csv
Saved 13 rows → c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\context_helps_CSC.csv
Saved 9 rows → c:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysi